
# Boston Housing Price Prediction – Model Training and Evaluation

## Objective
The objective of this notebook is to use our trained
Gradient Boosting model to predict house prices.

We will load the saved model (`best_boston_model.joblib`),
provide house-related input features, and generate
predicted house prices.

## Algorithms Used
1. Linear Regression
2. K-Nearest Neighbors (KNN) Regression
3. Support Vector Regression (SVR)
4. Decision Tree Regressor
5. Random Forest Regressor
6. Gradient Boosting Regressor
7. XGBoost Regressor

## Evaluation Metrics
- R² Score
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)

## Final Model

- Algorithm: Gradient Boosting Regressor
- R² Score: 0.9265
- MAE: 1.9231
- RMSE: 2.5723


In [2]:
!pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 10.7 MB/s eta 0:00:05
   - -------------------------------------- 2.1/48.9 MB 5.0 MB/s eta 0:00:10
   -- ------------------------------------- 3.1/48.9 MB 5.3 MB/s eta 0:00:09
   --- ------------------------------------ 4.7/48.9 MB 5.5 MB/s eta 0:00:09
   ---- ----------------------------------- 6.0/48.9 MB 5.5 MB/s eta 0:00:08
   ----- ---------------------------------- 7.1/48.9 MB 5.6 MB/s eta 0:00:08
   ------ --------------------------------- 8.4/48.9 MB 5.6 MB/s eta 0:00:08
   ------- -------------------------------- 9.4/48.9 MB 5.7 MB/s eta 0:00:07
   -------- ------------------------------- 10.7/48.9 MB 5.7 MB/s eta 0:00:07
   --------- ------------------------------ 12.1/48.9 MB 5.7 MB/s eta 0:00:07
   ---------- ----------------------------- 13.4/48.9 MB 5.7 MB/s eta 0:00:07
   ----------- ---------------------------- 14.7/48.9 MB 5.7 MB/s eta 0:00:06
 

In [3]:
from xgboost import XGBRegressor


## 1. Import Libraries

Import the required Python libraries for data handling, preprocessing, model training, and model evaluation.


In [4]:

# Data handling
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train-test split and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression algorithms
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

# Evaluation metrics
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')


What are these libraries for?
Library	Purpose
- NumPy, Pandas	=Data handling
- Matplotlib, Seaborn	=  Graphs
- StandardScaler	=  Feature scaling
- Pipeline	=  Preprocessing + model workflow
- sklearn models	=  Six regression algorithms
- XGBRegressor	=  XGBoost algorithm
- Metrics	=  Evaluate model performance


## 2. Load Dataset

Load the Boston Housing dataset using Pandas for model training and evaluation.


In [6]:

# Load dataset
df = pd.read_csv('boston_data.csv')

# Display first five rows
df.head()


,ID,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,1,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,2,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,4,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
3,5,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2
4,7,7.87,0,0.524,6.012,66.6,5.5605,5,311,15.2,12.43,22.9


In [7]:
print("Dataset Shape:", df.shape)

Dataset Shape: (333, 12)



## 3. Feature Selection

Separate the independent variables (features) and dependent variable (target).

- X: Input features used to predict house prices.
- y: Target variable (medv), representing house prices.

The ID column is excluded because it is only an identifier and does not provide useful information for prediction.


In [8]:

# Separate features and target
X = df.drop(columns=['ID', 'medv'])
y = df['medv']

# Check shapes
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


Features Shape: (333, 10)
Target Shape: (333,)



## 4. Train-Test Split

Split the dataset into training and testing sets.

- Training data (80%): Used to train the machine learning models.
- Testing data (20%): Used to evaluate model performance on unseen data.

A random state of 42 ensures reproducible results.


In [9]:

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Check dataset sizes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (266, 10)
X_test shape: (67, 10)
y_train shape: (266,)
y_test shape: (67,)



## 5. Model Preprocessing

Feature scaling is applied to Linear Regression, KNN, and SVR using StandardScaler.

Tree-based models do not require feature scaling. Pipelines help prevent data leakage by fitting preprocessing only on training data.


In [10]:

# Create preprocessing pipelines

models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor())
    ]),

    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR())
    ]),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(random_state=42),

    "Gradient Boosting": GradientBoostingRegressor(random_state=42),

    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )
}

print("All 7 models are initialized successfully.")


All 7 models are initialized successfully.



## 6. Model Training

Train all seven regression models using the training dataset.

Each model learns patterns and relationships between the input features and house prices.


In [11]:

# Train all models

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained successfully!")


Linear Regression trained successfully!
KNN trained successfully!
SVR trained successfully!
Decision Tree trained successfully!
Random Forest trained successfully!
Gradient Boosting trained successfully!
XGBoost trained successfully!



## 7. Model Evaluation

Evaluate the performance of all seven trained regression models using the test dataset.

The evaluation metrics used are:
- R² Score
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)


In [13]:

# Evaluate all trained models

results = []

for name, model in models.items():

    # Predict house prices
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Store results
    results.append({
        "Model": name,
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse
    })

# Create comparison DataFrame
results_df = pd.DataFrame(results)

# Sort models by R2 Score
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

results_df


,Model,R2 Score,MAE,RMSE
0,Gradient Boosting,0.927590,1.929107,2.552802
1,Random Forest,0.901263,2.207090,2.980973
2,XGBoost,0.898126,2.291726,3.027959
3,KNN,0.754683,3.025373,4.698738
4,Linear Regression,0.729173,3.770711,4.937002
5,Decision Tree,0.674613,3.443284,5.411500
6,SVR,0.614224,3.547978,5.892298


Gradient Boosting Regressor
- 0.9276  -  R² Score
- 1.9291  -  MAE
- 2.5528  -  RMSE

It explains approximately 92.76% of the variation in house prices on your test set.

### What Do These Results Tell Us?

- **Gradient Boosting:** Best current performance. It captures complex relationships in the dataset.
- **Random Forest:** Also performs well by combining multiple decision trees.
- **XGBoost:** Very close to Random Forest, but slightly lower in this experiment.
- **KNN:** Performs reasonably, but not as well as the boosting models.
- **Linear Regression:** Its performance suggests that a simple linear relationship may not capture all patterns.
- **Decision Tree:** A single tree may not generalize as well as ensemble models.
- **SVR:** Has the lowest R² score in this initial comparison.

**Note:** These results are from one train-test split, not proof that one model will always perform best.


## 8. Hyperparameter Tuning

Hyperparameter tuning is the process of finding the best configuration of a machine learning model to improve its performance.

In this project, RandomizedSearchCV will be used to search for suitable hyperparameters using 5-fold cross-validation.

The models will be compared using Root Mean Squared Error (RMSE).


In [15]:
from sklearn.model_selection import RandomizedSearchCV

### knn (k-Nearest Neighbour) using RandomizedSearchCV.

In [20]:
knn_params = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

knn_search = RandomizedSearchCV(
    estimator=models['KNN'],
    param_distributions=knn_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

knn_search.fit(X_train, y_train)

print("Best KNN Parameters:", knn_search.best_params_)
print("Best KNN CV RMSE:", -knn_search.best_score_)

Best KNN Parameters: {'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': 5}
Best KNN CV RMSE: 4.589170129275856


### We are telling Python which settings to try.
- n_neighbors: Try 3, 5, 7, 9, 11, 13 and 15 neighbours.
- weights:
  - uniform: Every neighbour has equal importance.
  - distance: Closer neighbours have more importance.
- p:
  - 1: Manhattan distance.
  - 2: Euclidean distance.


Parameter	Meaning
- estimator	-  The KNN pipeline we want to tune
- param_distributions	-   Hyperparameter values to try
- n_iter=20	  -   Try 20 randomly selected combinations
- scoring  -   Evaluate using RMSE
- cv=5	-   Perform 5-fold cross-validation
- random_state=42  -	Keep random selection reproducible
- n_jobs=-1	  - Use available CPU cores

- It means that the best KNN configuration found by RandomizedSearchCV achieved an average RMSE of approximately 4.59 across the 5 cross-validation folds.

- Your earlier KNN test RMSE was 4.6987. These scores are close, but they come from different evaluation methods, so don't treat them as a direct final comparison.

### SVR (Support Vector Regression) using RandomizedSearchCV.

In [21]:
svr_params = {
    'model__kernel': ['linear', 'rbf'],
    'model__C': [0.1, 1, 10, 50, 100],
    'model__epsilon': [0.01, 0.1, 0.2, 0.5],
    'model__gamma': ['scale', 'auto']
}

svr_search = RandomizedSearchCV(
    estimator=models['SVR'],
    param_distributions=svr_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

svr_search.fit(X_train, y_train)

print("Best SVR Parameters:", svr_search.best_params_)
print("Best SVR CV RMSE:", -svr_search.best_score_)

Best SVR Parameters: {'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__epsilon': 0.2, 'model__C': 100}
Best SVR CV RMSE: 3.7612083139051955


- Your tuned SVR achieved an average CV RMSE of 3.76, compared with its initial test RMSE of 5.89.

- This suggests that tuning found a more promising configuration for SVR. We'll confirm its actual test performance later.

### Hyperparameter tuning for Decision Tree

In [22]:
dt_params = {
    'max_depth': [None, 3, 5, 7, 10, 15], # Maximum depth of the tree
    'min_samples_split': [2, 5, 10],  #Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4],    # Minimum samples required in a leaf node
    'max_features': [None, 'sqrt']    # Number of features considered when finding a split
}

dt_search = RandomizedSearchCV(
    estimator=models['Decision Tree'],
    param_distributions=dt_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

dt_search.fit(X_train, y_train)

print("Best Decision Tree Parameters:", dt_search.best_params_)
print("Best Decision Tree CV RMSE:", -dt_search.best_score_)

Best Decision Tree Parameters: {'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 15}
Best Decision Tree CV RMSE: 4.3762913642877574


### Hyperparameter tuning for Random Forest

- n_estimators: Number of trees.

- max_depth: Maximum depth of each tree.

- min_samples_split: Minimum samples required to split a node.

- min_samples_leaf: Minimum samples required in a leaf.

- max_features: Number of features considered for each split.

In [23]:
rf_params = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 5, 8, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [1.0, 'sqrt']
}

rf_search = RandomizedSearchCV(
    estimator=models['Random Forest'],
    param_distributions=rf_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best Random Forest Parameters:", rf_search.best_params_)
print("Best Random Forest CV RMSE:", -rf_search.best_score_)

Best Random Forest Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 10}
Best Random Forest CV RMSE: 3.595165012555422


#### 1. Initial RMSE = 2.9809
In your initial evaluation:
- You divided the dataset into 80% training and 20% testing.
- The model learned from 266 training records.
- It was evaluated on 67 test records.
- RMSE was 2.9809.

#### 2. Cross-validation RMSE = 3.5952
During hyperparameter tuning:
- RandomizedSearchCV uses only the 266 training records.
- It divides those records into 5 folds.
- It trains and evaluates the model five times for each selected parameter combination.
- It calculates the average RMSE across the folds.
- The best average CV RMSE was 3.5952.
#### 3. Why is CV RMSE higher?
Because the model is evaluated on different subsets of data.

#### 4. Which RMSE should we trust?
Both are useful, but for different purposes:
- CV RMSE: Helps select hyperparameters.
- Test RMSE: Helps evaluate final model performance on unseen data

##### We should not choose a model just because its initial test RMSE is lower.


## Hyperparameter tuning for Gradient Boosting

In [24]:
gb_params = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.8, 1.0]
}

gb_search = RandomizedSearchCV(
    estimator=models['Gradient Boosting'],
    param_distributions=gb_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

gb_search.fit(X_train, y_train)

print("Best Gradient Boosting Parameters:", gb_search.best_params_)
print("Best Gradient Boosting CV RMSE:", -gb_search.best_score_)

Best Gradient Boosting Parameters: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 3, 'learning_rate': 0.03}
Best Gradient Boosting CV RMSE: 3.4855022238385374


## Hyperparameter tuning for XGBoost

In [25]:
xgb_params = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5, 10]
}

xgb_search = RandomizedSearchCV(
    estimator=models['XGBoost'].set_params(n_jobs=1),
    param_distributions=xgb_params,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=5,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)

print("Best XGBoost Parameters:", xgb_search.best_params_)
print("Best XGBoost CV RMSE:", -xgb_search.best_score_)

Best XGBoost Parameters: {'subsample': 0.8, 'reg_lambda': 5, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best XGBoost CV RMSE: 3.545994198298833


## 3. Final Comparison of All Six Tuned Models

**Hyperparameter tuning completed successfully!**

| Model | Best CV RMSE |
|---|---:|
| Gradient Boosting | **3.4855** |
| XGBoost | 3.5460 |
| Random Forest | 3.5952 |
| SVR | 3.7612 |
| Decision Tree | 4.3763 |
| KNN | 4.5892 |

**Current Observation:** Gradient Boosting has the lowest CV RMSE among the six tuned models.

This is our best model so far based on cross-validation, but we still need to test all tuned models on the same untouched test dataset.

# Remember, lower RMSE means better performance.

In [26]:
best_models = {
    "KNN": knn_search.best_estimator_,
    "SVR": svr_search.best_estimator_,
    "Decision Tree": dt_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_,
    "Gradient Boosting": gb_search.best_estimator_,
    "XGBoost": xgb_search.best_estimator_
}

print("All 6 tuned models are ready for final evaluation.")

All 6 tuned models are ready for final evaluation.


In [29]:
print(best_models)

{'KNN': Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsRegressor(p=1, weights='distance'))]), 'SVR': Pipeline(steps=[('scaler', StandardScaler()),
                ('model', SVR(C=100, epsilon=0.2))]), 'Decision Tree': DecisionTreeRegressor(max_depth=15, min_samples_leaf=2, min_samples_split=10,
                      random_state=42), 'Random Forest': RandomForestRegressor(max_depth=10, min_samples_split=10, random_state=42), 'Gradient Boosting': GradientBoostingRegressor(learning_rate=0.03, min_samples_split=5,
                          n_estimators=200, random_state=42, subsample=0.8), 'XGBoost': XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance

In [30]:
final_results = []

for name, model in best_models.items():

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    final_results.append({
        "Model": name,
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse
    })

final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.sort_values(
    by="RMSE", ascending=True
).reset_index(drop=True)

final_results_df

,Model,R2 Score,MAE,RMSE
0,Gradient Boosting,0.926479,1.923063,2.572301
1,Random Forest,0.909045,2.148235,2.861077
2,XGBoost,0.893391,2.113951,3.097517
3,Decision Tree,0.793709,2.893290,4.308810
4,SVR,0.749218,2.675494,4.750785
5,KNN,0.559787,3.380133,6.294316


#### Remember:
- Higher R² is better.
- Lower MAE is better.
- Lower RMSE is better.

### 1. What Do These Results Tell Us?

- **Gradient Boosting:** Best performance. It has the highest R² and lowest MAE and RMSE.
- **Random Forest:** Second-best model, with strong prediction performance.
- **XGBoost:** Also performs well, but slightly below Random Forest.
- **Decision Tree:** Performs reasonably but has higher prediction errors.
- **SVR:** Performs better than KNN but has relatively high RMSE.
- **KNN:** Lowest performance among the six tuned models.

### 2. Why Is Gradient Boosting Our Final Model?

**ACCURACY: 92.65%**

**R² Score: 0.9265**

**MAE: 1.9231**

**RMSE: 2.5723**

Gradient Boosting is our best model because:

1. It explains approximately 92.65% of the variation in house prices on the test set.
2. It has the lowest prediction error.
3. It performs better than the other five tuned models on this test split.

### 3. Final Model Selection Conclusion

After hyperparameter tuning and final evaluation, Gradient Boosting was selected as the best-performing model.

It achieved an R² Score of 0.9265, MAE of 1.9231, and RMSE of 2.5723 on the test dataset.

The results indicate that Gradient Boosting performed better than the other five models in this experiment.

Therefore, Gradient Boosting will be used as the final model for house price prediction.

## Permutation Feature Importance

Permutation Feature Importance helps us understand which features
contribute most to the prediction performance of our final model.

It works by randomly shuffling one feature at a time and measuring
how much the model's performance decreases.

In [38]:
from sklearn.inspection import permutation_importance

# Calculate feature importance
final_model = best_models["Gradient Boosting"]

perm_importance = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

# Create a DataFrame
importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm_importance.importances_mean,
    "Std": perm_importance.importances_std
})

# Sort features by importance
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

display(importance_df)

,Feature,Importance,Std
9,lstat,6.078234,0.715616
3,rm,3.762950,0.377108
5,dis,1.489932,0.606955
2,nox,0.614711,0.129303
8,ptratio,0.525134,0.092286
4,age,0.193840,0.073270
7,tax,0.129385,0.054215
1,chas,0.042961,0.034983
6,rad,0.012252,0.038263
0,indus,-0.031638,0.048798


### What does this mean?
- lstat has the highest importance. Shuffling it increases the model's prediction error considerably.
- rm is the second most important feature.
- dis also contributes to prediction performance.
- indus has slightly negative importance (-0.0316). This means shuffling it slightly improved the score in this experiment. It does not mean the feature has negative real-world importance.

# Save the best model

In [39]:
import joblib

In [40]:
best_model = best_models['Gradient Boosting']

joblib.dump(best_model, 'best_boston_model.joblib')

print("Best model saved successfully!")

Best model saved successfully!
